In [ ]:
import numpy as np
import pylab as pl
import pandas as pd
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import cross_val_score, GridSearchCV
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print(os.listdir("../input"))

# Any results you write to the current directory are saved as output.

In [ ]:
train= pd.read_csv('../input/used-cars-price-prediction/train-data.csv')
train.head()

# Visualization

In [ ]:
f, axes = plt.subplots(1,1, figsize = (16, 5))
g1 = sns.distplot(train["Price"], color="blue",ax = axes)
plt.title("Distributional of price")

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go

ep = train['Owner_Type'].value_counts().reset_index()
ep.columns = [
    'Owner_Type', 
    'percent'
]
ep['percent'] /= len(train)

fig = px.pie(
    ep, 
    names='Owner_Type', 
    values='percent', 
    title='Countplot of Owner_Type', 
    width=800,
    height=500 
)

fig.show()

In [ ]:
cdi = train.sort_values(by='Price', ascending=False)[:100]
figure = plt.figure(figsize=(10,6))
sns.barplot(y=cdi.Name, x=cdi.Price)
plt.xticks()
plt.xlabel('Price')
plt.ylabel('Name')
plt.title('Name (Cars) by Price')
plt.show()

In [ ]:

plt.figure(figsize=[15,4])
sns.countplot(x='Transmission', hue='Owner_Type',edgecolor="black", alpha=0.7, data=train)
sns.despine()
plt.title("Countplot of transmission by Owner_Type ")
plt.show()

In [ ]:
sns.barplot(train["Owner_Type"],train["Kilometers_Driven"],hue= train["Transmission"],palette="spring")
plt.xticks(rotation=80)
plt.title("Transmission: Kilometers_Driven comparsion")

In [ ]:
sns.barplot(train["Owner_Type"],train["Kilometers_Driven"],hue= train["Location"],palette="spring")
plt.xticks(rotation=80)
plt.title("Location: Kilometers_Driven comparsion")

# Owner_Type = First

In [ ]:
display(train[train["Owner_Type"]=="First"][["Name","Location","Transmission","Year","Kilometers_Driven","Fuel_Type","Mileage","Engine","Power",
                                       "Price"]].sort_values(by="Price", ascending= False).head(10).style.background_gradient(cmap="spring"))

# Owner_Type = Second

In [ ]:
display(train[train["Owner_Type"]=="Second"][["Name","Location","Transmission","Year","Kilometers_Driven","Fuel_Type","Mileage","Engine","Power",
                                       "Price"]].sort_values(by="Price", ascending= False).head(10).style.background_gradient(cmap="spring"))

# Owner_Type = Third

In [ ]:
display(train[train["Owner_Type"]=="Third"][["Name","Location","Transmission","Year","Kilometers_Driven","Fuel_Type","Mileage","Engine","Power",
                                       "Price"]].sort_values(by="Price", ascending= False).head(10).style.background_gradient(cmap="spring"))

# Owner_Type =Fourth & Above

In [ ]:
display(train[train["Owner_Type"]=="Fourth & Above"][["Name","Location","Transmission","Year","Kilometers_Driven","Fuel_Type","Mileage","Engine","Power",
                                       "Price"]].sort_values(by="Price", ascending= False).head(10).style.background_gradient(cmap="spring"))

# Transmission : Automatic


In [ ]:
display(train[train["Transmission"]=="Automatic"][["Name","Location","Transmission","Year","Kilometers_Driven","Fuel_Type","Mileage","Engine","Power",
                                       "Price"]].sort_values(by="Price", ascending= False).head(30).style.background_gradient(cmap="spring"))

# Transmission : Manual

In [ ]:
display(train[train["Transmission"]=="Manual"][["Name","Location","Transmission","Year","Kilometers_Driven","Fuel_Type","Mileage","Engine","Power",
                                       "Price"]].sort_values(by="Price", ascending= False).head(30).style.background_gradient(cmap="spring"))

In [ ]:
test= pd.read_csv('../input/used-cars-price-prediction/test-data.csv')
test.head()

# Prepocessing

In [ ]:
labels = {}
for col in train.select_dtypes(exclude = np.number).columns.tolist():
    le = LabelEncoder().fit(pd.concat([train[col].astype(str),test[col].astype(str)]))   
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
    labels [col] = le
print('Categorical columns:', list(labels.keys()))

# Check Missing Data

In [ ]:
print(f'Percent of Nans in Train Data : {round(train.isna().sum().sum()/len(train), 2)}')
print(f'Percent of Nans in Test  Data : {round(test.isna().sum().sum()/len(test), 2)}')

In [ ]:
train = train.replace([np.inf, -np.inf], np.nan)
train= train.fillna(train.mean())
train

In [ ]:
test = test.replace([np.inf, -np.inf], np.nan)
test= test.fillna(test.mean())
test

Eliminate irrelevant variables in analysis such as New_Price.

In [ ]:
train = train.drop(columns=['New_Price'],
                 axis=1)
train = train.dropna(how='any')
print(train.shape)

In [ ]:
test= test.drop(columns=['New_Price'],

                 axis=1)
test = test.dropna(how='any')
print(test.shape)

In [ ]:
#Select feature column names and target variable we are going to use for training

features =['Name','Location','Year','Kilometers_Driven','Fuel_Type','Transmission','Owner_Type','Mileage','Engine','Power','Seats']
target = 'Price'

In [ ]:
#This is input which our classifier will use as an input.
train[features].head(10)

In [ ]:
#This is input which our classifier will use as an input.
test[features].head(10)

# Model

## Data for training and validation (Measure MSE)

To select a set of training data that will be input in the Machine Learning algorithm, to ensure that the classification algorithm training can be generalized well to new data. For this study using a sample size of 15%, assumed it ideal ratio between training and validation

In [ ]:
from sklearn.model_selection import train_test_split
Y = train['Price']
X = train.drop(columns=['Price'])
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15, random_state=9)

print('X train shape: ', X_train.shape)
print('Y train shape: ', Y_train.shape)
print('X test shape: ', X_test.shape)
print('Y test shape: ', Y_test.shape)

# GridSearchCV: RandomForestRegressor

<img src='https://media.geeksforgeeks.org/wp-content/uploads/20200516180708/Capture482.png' width='400'>



* GridSearchCV is a library function that is a member of sklearn's model_selection package. It helps to loop through predefined hyperparameters and fit your estimator (model) on your training set. So, in the end, you can select the best parameters from the listed hyperparameters. Reference : https://towardsdatascience.com/grid-search-for-hyperparameter-tuning-9f63945e8fec#:~:text=What%20is%20GridSearchCV%3F,parameters%20from%20the%20listed%20hyperparameters.

* A Random Forest is an ensemble technique capable of performing both regression and classification tasks with the use of multiple decision trees and a technique called Bootstrap and Aggregation, commonly known as bagging. The basic idea behind this is to combine multiple decision trees in determining the final output rather than relying on individual decision trees. Random Forest has multiple decision trees as base learning models. We randomly perform row sampling and feature sampling from the dataset forming sample datasets for every model. This part is called Bootstrap.
Reference : https://www.geeksforgeeks.org/random-forest-regression-in-python/


# Model 1

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.metrics import mean_absolute_error 
# We define the model
estimator = RandomForestRegressor(random_state = 42,criterion='mse')
para_grids = {
            "n_estimators" : [10,50,100],
            "max_features" : ["auto", "log2", "sqrt"],
            'max_depth' : [4,5,6,7,8,9,15],
            "bootstrap"    : [True, False]
        }


Grid = GridSearchCV(estimator, para_grids,cv= 5)
# We train model
Grid.fit(X_train, Y_train)
best_param = Grid.best_estimator_
print(best_param)


## Prediction

In [ ]:
# We predict target values (Split 15% from training data)
Y_predict = best_param.predict(X_test)
Y_predict

# Model 2

In [ ]:
model = RandomForestRegressor(bootstrap=False, max_depth=15, max_features='log2',
                      random_state=42)
# We train model
model.fit(train[features],train[target])

## prediction 

In [ ]:
#Make predictions using the features from the test data set
predictions = model.predict(test[features])


# Model 3

k-fold is a popular kind of cross-validation technique, in which, say k=10 for example, 9 folds for training and 1 fold for testing purpose and this repeats unless all folds get a chance to be the test set one by one. This way, it provides a good idea of the generalization ability of the model, especially when we have limited data and can't afford to split into test and training data.

Reference :https://www.researchgate.net/post/What-is-the-purpose-of-performing-cross-validation

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from numpy import mean, std

cv = KFold(n_splits=10, random_state=1, shuffle=True)
scores = cross_val_score(model, train[features], train[target], cv=cv, n_jobs=-1)
print('Mean of Scores: %.3f' % (mean(scores)))

# Measure mean squared error 

In statistics, the mean squared error (MSE) or mean squared deviation (MSD) of an estimator (of a procedure for estimating an unobserved quantity) measures the average of the squares of the errors—that is, the average squared difference between the estimated values and the actual value.

Reference : https://en.wikipedia.org/wiki/Mean_squared_error





In [ ]:
#Model 1
from sklearn.metrics import mean_squared_error
mean_squared_error(Y_test, Y_predict)


# Make Submission

In [ ]:
#Create a  DataFrame
submission = pd.DataFrame({'Owner_Type':test['Owner_Type'],'Price':predictions})                        

#Visualize the first 10 rows
submission.head(10)

In [ ]:
#Convert DataFrame to a csv file that can be uploaded
#This is saved in the same directory as your notebook
filename = 'submission.csv'

submission.to_csv(filename,index=True)

print('Saved file: ' + filename)